In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [7]:
data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

In [8]:
# Scale pm2_5
scaler = StandardScaler()
pm25_scaled = scaler.fit_transform(data[['pm2_5']])

# Create sliding windows
def create_windows(df, window=12):
    X, y = [], []
    for i in range(len(df) - window):
        X.append(df[i : i+window])
        y.append(df[i+window])
    return np.array(X), np.array(y)

X, y = create_windows(pm25_scaled, window=12)
# X shape: (n-12, 12, 1)
# y shape: (n-12, 1)

# Temporal split
split = int(np.ceil(0.8 * len(X)))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time

torch.manual_seed(42)
np.random.seed(42)

# --- Hyperparameters ---
HIDDEN = 64
LAYERS = 2
LR     = 0.001
EPOCHS = 10
BATCH  = 64

# --- Convert to tensors ---
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

# --- DataLoader ---
loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                    batch_size=BATCH, shuffle=False)

# --- Model ---
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm   = nn.LSTM(input_size=1, hidden_size=HIDDEN,
                              num_layers=LAYERS, batch_first=True)
        self.linear = nn.Linear(HIDDEN, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])  # take last timestep only

model     = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# --- Training loop ---
train_start = time.time()

for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss.item():.4f}")

train_time = time.time() - train_start

Epoch 1/10 - Loss: 0.0384
Epoch 2/10 - Loss: 0.0269
Epoch 3/10 - Loss: 0.0082
Epoch 4/10 - Loss: 0.0059
Epoch 5/10 - Loss: 0.0057
Epoch 6/10 - Loss: 0.0056
Epoch 7/10 - Loss: 0.0056
Epoch 8/10 - Loss: 0.0056
Epoch 9/10 - Loss: 0.0056
Epoch 10/10 - Loss: 0.0056


In [10]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# --- Predict ---
model.eval()

with torch.no_grad():
    train_preds_scaled = model(X_train_t).numpy()

inference_start = time.time()
with torch.no_grad():
    test_preds_scaled = model(X_test_t).numpy()
inference_time = (time.time() - inference_start) / len(X_test)

# --- Inverse transform ---
train_preds  = scaler.inverse_transform(train_preds_scaled)
test_preds   = scaler.inverse_transform(test_preds_scaled)
y_train_orig = scaler.inverse_transform(y_train)
y_test_orig  = scaler.inverse_transform(y_test)

# --- Metrics ---
train_rmse = root_mean_squared_error(y_train_orig, train_preds)
train_mae  = mean_absolute_error(y_train_orig, train_preds)
train_r2   = r2_score(y_train_orig, train_preds)

test_rmse = root_mean_squared_error(y_test_orig, test_preds)
test_mae  = mean_absolute_error(y_test_orig, test_preds)
test_r2   = r2_score(y_test_orig, test_preds)

print(f"Train RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2: {test_r2:.4f}")
print(f"Training Time:  {train_time:.2f}s")
print(f"Inference Time: {inference_time:.6f}s per sample")

Train RMSE: 3.8448 | MAE: 2.1953 | R2: 0.9360
Test  RMSE: 7.9390 | MAE: 3.8478 | R2: 0.9295
Training Time:  31.80s
Inference Time: 0.000017s per sample
